# yt-clips — One-Click Colab Worker
Runtime → T4 GPU.  Run cell below, wait ~2min, done.

**Before starting:**
- Add `OPENROUTER_API_KEY` to Colab Secrets (top-left key icon)
- Or upload `.env` to `yt-clips/` on Google Drive

In [ ]:
# CELL 1: ONE CLICK — everything below runs automatically
import os, subprocess, sys, time, json, threading
from pathlib import Path

def run(cmd, timeout=120):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0:
        err = (r.stderr.strip()[-200:] if r.stderr else r.stdout.strip()[-200:])
        print(f"  \u26a0 {cmd[:60]}... {err}")
    return r

# ── 1. Mount Drive ──
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
for p in ["/content/drive/MyDrive/yt-clips", "/content/drive/My Drive/yt-clips"]:
    if Path(p).exists():
        os.chdir(p); break
print(f"Working dir: {os.getcwd()}")

# ── 2. Load secrets (Colab Secrets > .env) ──
if Path(".env").exists():
    for line in open(".env"):
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip()
try:
    from google.colab import userdata
    for k in ["OPENROUTER_API_KEY", "GROQ_API_KEY", "GOOGLE_API_KEY"]:
        v = userdata.get(k)
        if v: os.environ[k] = v
except: pass

# ── 3. Force-pull latest code ──
if Path(".git").exists():
    run("git stash 2>/dev/null", timeout=10)
    run("git fetch origin main 2>&1", timeout=30)
    run("git reset --hard origin/main 2>&1", timeout=15)
else:
    print("Not a git repo — cloning fresh...")
    run(f"git clone https://github.com/fearless16/yt-clips.git /tmp/yt-clips 2>&1", timeout=60)
    os.chdir("/tmp/yt-clips")
    print(f"Working dir: {os.getcwd()}")
sys.path.insert(0, ".")

# ── Verify automation module ──
if not Path("automation/__init__.py").exists():
    raise RuntimeError("automation/ not found after pull. Check Colab Drive setup.")

# ── 4. Install deps ──
print("Installing system deps...")
run("apt-get update -qq && apt-get install -y -qq aria2 ffmpeg > /dev/null 2>&1")
print("Installing torch+cu121 (T4)...")
run("pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121", timeout=300)
print("Installing pipeline deps...")
run("pip install -q yt-dlp faster-whisper rich PyYAML opencv-python-headless numpy "
    "filterpy scipy openai python-dotenv Pillow requests "
    "ultralytics gfpgan basicsr realesrgan "
    "google-api-python-client google-auth-httplib2 google-auth-oauthlib")
try:
    import utils.torchvision_compat
except: pass

# ── 5. Verify automation import ──
import automation
print(f"automation v{automation.VERSION} OK")

# ── 6. Verify GPU ──
import torch
print(f"CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")

# ── 7. Create dirs + kill stale ──
for d in ["input","temp","transcripts","highlights","shorts","logs","photos"]:
    Path(d).mkdir(exist_ok=True)
run("pkill -f 'python watcher.py' 2>/dev/null; pkill -f serveo 2>/dev/null; fuser -k 5000/tcp 2>/dev/null; sleep 2")

# ── 8. Start Watcher (nohup daemon) ──
run(f"nohup {sys.executable} watcher.py > watcher.log 2>&1 &")
time.sleep(4)
alive = subprocess.run("curl -sf http://localhost:5000/health", shell=True, capture_output=True).returncode == 0
print(f"Watcher: {'OK' if alive else 'FAILED'}")
if not alive:
    print(open("watcher.log").read().strip()[-300:])

# ── 9. Start Tunnel (serveo → localhost.run fallback) ──
def try_tunnel(cmd, label, wait=8):
    run(cmd.format(LOG="tunnel.log"), timeout=15)
    time.sleep(wait)
    if Path("tunnel.log").exists():
        for line in open("tunnel.log"):
            for w in line.split():
                w = w.strip().rstrip(",.;")
                if w.startswith("https://"):
                    Path("colab_url.txt").write_text(w)
                    return w
    return None

url = try_tunnel("nohup ssh -o StrictHostKeyChecking=no -o ServerAliveInterval=30 -R 80:localhost:5000 serveo.net > {LOG} 2>&1 &", "serveo")
if not url:
    print("serveo failed, trying localhost.run...")
    run("pkill -f serveo 2>/dev/null; sleep 2")
    url = try_tunnel("nohup ssh -o StrictHostKeyChecking=no -o ServerAliveInterval=30 -R 80:localhost:5000 nokey@localhost.run > {LOG} 2>&1 &", "localhost.run", wait=12)

print(f"Tunnel: {url or 'N/A (Drive API fallback)'}")

# ── 10. Done ──
print("\n" + "="*55)
print("  ONE-CLICK SETUP COMPLETE")
print("="*55)
print()
print("From your Mac, send jobs:")
print('  python bridge.py "https://youtu.be/..."')
print()
print("Or run pipeline directly in this cell below:")
print('  !python pipeline.py "https://youtu.be/..."')

In [ ]:
# CELL 2: Dashboard — keep this cell running to hold the session
from automation.memory import memory_report
from automation.env import gpu_info
from automation.tunnel import tunnel_status

ts = tunnel_status()
print("="*55)
print("  WORKER ONLINE — live updates every 15s")
print("="*55)

last_pos = 0
while True:
    time.sleep(15)
    log_path = Path("watcher.log")
    if log_path.exists():
        with open(log_path) as f:
            f.seek(last_pos)
            for l in f:
                if l.strip(): print(l.strip())
            last_pos = f.tell()
    mem = memory_report()
    gpu = gpu_info()
    print(f"[MEM {mem.get('free_gb',0):.1f}GB  GPU {gpu.get('name','?')[:20]} {gpu.get('memory_free_gb',0):.1f}GB  "
          f"Tunnel {'OK' if Path('colab_url.txt').exists() else '...'}]")